# NB05 - Generator: FLUX.1-schnell  (`flux_schnell`)

**Run on its own Kaggle account, in parallel with the other generators.** Reads the frozen captions from NB02, generates one AI image per real image with **FLUX.1-schnell**, applies the *same* canonical preprocess used for the real images, and writes to `data/flux_schnell/`.

### Prerequisites
- NB00, NB01, NB02 finished (repo, library, config, real images, captions all exist).
- **Internet: ON. GPU: ON** (T4 x2 is fine).
- HF write token as the Kaggle secret `HF_TOKEN` on this account.

### Parallel-safe by design
- Writes only to `data/flux_schnell/` - never collides with the other five generators.
- Resumes from its own shards (counts what exists; `progress.json` is never trusted for counts).
- Seed = `hash(flux_schnell, source_real_id)`, so a crash/retry/double-run reproduces **identical** bytes.
- Commits one batch every 20 minutes; stops at exactly {target} accepted images.

To run the whole set: open this notebook on one account, and NB03-NB08 each on a different account. They can start on different days; NB09 only needs all six eventually finished.

In [ ]:
import sys, subprocess
def pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *pkgs], check=True)
# diffusers stack + hub/io. sentencepiece+protobuf are needed by the T5 text encoders (Flux/PixArt).
pip("diffusers>=0.30", "transformers>=4.40", "accelerate", "safetensors",
    "sentencepiece", "protobuf", "huggingface_hub>=0.23", "pyarrow", "pillow")
print("deps installed")

In [ ]:
REPO_ID = "Shanmuk4622/ai-image-detection-dataset"          # same dataset repo as NB00
MODEL   = "flux_schnell"            # <-- this is the ONLY line that differs between NB03-NB08

import os
from huggingface_hub import hf_hub_download
def load_token():
    try:
        from kaggle_secrets import UserSecretsClient
        t = UserSecretsClient().get_secret("HF_TOKEN")
        if t: return t.strip()
    except Exception:
        pass
    for k in ("HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        if os.environ.get(k): return os.environ[k].strip()
    import getpass
    return getpass.getpass("HF write token: ").strip()
TOKEN = load_token()

# download + import the SAME shared library NB00 uploaded (identical canonical preprocess)
lib = hf_hub_download(REPO_ID, "ai_detect_common.py", repo_type="dataset", token=TOKEN)
import importlib.util
spec = importlib.util.spec_from_file_location("ai_detect_common", lib)
C = importlib.util.module_from_spec(spec); spec.loader.exec_module(C)
cfg = C.read_config(REPO_ID, TOKEN)
assert MODEL in cfg["generators"], f"{MODEL} not in config generators {list(cfg['generators'])}"
g = cfg["generators"][MODEL]
print("model:", MODEL, "| spec:", g)

In [ ]:
import torch
from diffusers import (StableDiffusionPipeline, StableDiffusionXLPipeline, FluxPipeline,
                       AutoPipelineForText2Image, PixArtSigmaPipeline,
                       StableCascadePriorPipeline, StableCascadeDecoderPipeline)

DTYPE = torch.float16   # T4-friendly; used for every model here

def first(x):           # config stores cascade's steps/guidance as [prior, decoder]
    return x[0] if isinstance(x, (list, tuple)) else x

def build_generator(model_key, g):
    """Returns gen(prompt, seed) -> PIL.Image at native resolution.
    A CPU generator is used for ALL models so results are reproducible regardless
    of the CPU-offload mode each pipeline uses."""
    mid, native, steps, guid = g["model_id"], g["native"], g["steps"], g["guidance"]

    if model_key == "sd15":
        pipe = StableDiffusionPipeline.from_pretrained(
            mid, torch_dtype=DTYPE, safety_checker=None, requires_safety_checker=False)
        pipe = pipe.to("cuda")
        try: pipe.enable_vae_slicing()
        except Exception: pass
        pipe.set_progress_bar_config(disable=True)
        def gen(prompt, seed):
            gn = torch.Generator(device="cpu").manual_seed(seed)
            return pipe(prompt, height=native, width=native, num_inference_steps=steps,
                        guidance_scale=guid, generator=gn).images[0]
        return gen

    if model_key == "sdxl":
        pipe = StableDiffusionXLPipeline.from_pretrained(
            mid, torch_dtype=DTYPE, use_safetensors=True, add_watermarker=False)
        pipe.enable_model_cpu_offload()
        try: pipe.enable_vae_slicing()
        except Exception: pass
        pipe.set_progress_bar_config(disable=True)
        def gen(prompt, seed):
            gn = torch.Generator(device="cpu").manual_seed(seed)
            return pipe(prompt=prompt, height=native, width=native, num_inference_steps=steps,
                        guidance_scale=guid, generator=gn).images[0]
        return gen

    if model_key == "flux_schnell":
        pipe = FluxPipeline.from_pretrained(mid, torch_dtype=DTYPE)
        pipe.enable_model_cpu_offload()
        try: pipe.vae.enable_slicing(); pipe.vae.enable_tiling()
        except Exception: pass
        pipe.set_progress_bar_config(disable=True)
        def gen(prompt, seed):
            gn = torch.Generator(device="cpu").manual_seed(seed)
            return pipe(prompt=prompt, height=native, width=native, num_inference_steps=steps,
                        guidance_scale=guid, max_sequence_length=256, generator=gn).images[0]
        return gen

    if model_key == "kandinsky22":
        pipe = AutoPipelineForText2Image.from_pretrained(mid, torch_dtype=DTYPE)
        pipe.enable_model_cpu_offload()
        pipe.set_progress_bar_config(disable=True)
        def gen(prompt, seed):
            gn = torch.Generator(device="cpu").manual_seed(seed)
            return pipe(prompt=prompt, height=native, width=native, num_inference_steps=steps,
                        guidance_scale=guid, generator=gn).images[0]
        return gen

    if model_key == "pixart_sigma":
        pipe = PixArtSigmaPipeline.from_pretrained(mid, torch_dtype=DTYPE, use_safetensors=True)
        pipe.enable_model_cpu_offload()
        pipe.set_progress_bar_config(disable=True)
        def gen(prompt, seed):
            gn = torch.Generator(device="cpu").manual_seed(seed)
            return pipe(prompt=prompt, height=native, width=native, num_inference_steps=steps,
                        guidance_scale=guid, generator=gn).images[0]
        return gen

    if model_key == "sd_cascade":
        prior = StableCascadePriorPipeline.from_pretrained(g["prior_id"], torch_dtype=DTYPE)
        decoder = StableCascadeDecoderPipeline.from_pretrained(mid, torch_dtype=DTYPE)
        prior.enable_model_cpu_offload(); decoder.enable_model_cpu_offload()
        prior.set_progress_bar_config(disable=True); decoder.set_progress_bar_config(disable=True)
        p_steps, d_steps = steps[0], steps[1]
        p_guid,  d_guid  = guid[0],  guid[1]
        def gen(prompt, seed):
            gn = torch.Generator(device="cpu").manual_seed(seed)
            po = prior(prompt=prompt, height=native, width=native, negative_prompt="",
                       guidance_scale=p_guid, num_inference_steps=p_steps,
                       num_images_per_prompt=1, generator=gn)
            return decoder(image_embeddings=po.image_embeddings, prompt=prompt, negative_prompt="",
                           guidance_scale=d_guid, num_inference_steps=d_steps,
                           output_type="pil", generator=gn).images[0]
        return gen

    raise ValueError(f"unknown model_key {model_key}")

generate = build_generator(MODEL, g)
print("pipeline ready for", MODEL)

In [ ]:
import gc, torch
api = C.HfApi(token=TOKEN)
TARGET = cfg["target_per_generator"]
native = g["native"]
g_steps = int(g["steps"][0]) if isinstance(g["steps"], (list, tuple)) else int(g["steps"])
g_guid  = float(g["guidance"][0]) if isinstance(g["guidance"], (list, tuple)) else float(g["guidance"])

# 1. load the FROZEN captions table (written by NB02) -> {source_real_id: caption}
captions = {}
for f in C.list_shards(REPO_ID, "captions/", TOKEN):
    local = C.hf_hub_download(REPO_ID, f, repo_type="dataset", token=TOKEN)
    t = C.pq.read_table(local, columns=["source_real_id", "caption"])
    for sid, cap in zip(t.column("source_real_id").to_pylist(), t.column("caption").to_pylist()):
        captions[sid] = cap
print("captions available:", len(captions))
assert captions, "No captions found - run NB02 first."

# 2. resume: which source_real_ids this model already generated (authoritative = the shards)
done = C.reconcile_ids(REPO_ID, f"data/{MODEL}/", TOKEN)
todo = sorted(set(captions) - done)          # sorted -> deterministic order across sessions
print(f"already done: {len(done)} | remaining: {len(todo)} | target: {TARGET}")

# 3. generate
writer = C.ShardWriter(api, REPO_ID, f"data/{MODEL}/",
                       commit_interval=cfg["commit_interval_seconds"],
                       max_rows=cfg["batch_size"], token=TOKEN)
accepted, failed = 0, 0
try:
    for i, sid in enumerate(todo):
        if len(done) + accepted >= TARGET:
            print("target reached."); break
        prompt = captions[sid]
        seed = C.make_seed(MODEL, sid)        # deterministic: re-runs reproduce identical bytes
        try:
            img = generate(prompt, seed)               # native-resolution PIL
            png = C.canonical_preprocess(img)          # SAME transform as the real images -> 512 PNG
        except Exception as e:
            failed += 1
            if failed <= 20 or failed % 50 == 0:
                print(f"  gen failed for {sid} ({type(e).__name__}: {e}); will retry on a later run")
            continue
        num = str(sid).split("_")[-1]                  # real_000123 -> 000123
        row = C.empty_row()
        row.update(dict(image_id=f"{MODEL}_{num}", source_real_id=sid, label=1, generator=MODEL,
                        source_dataset=MODEL, prompt=prompt, image=png,
                        width=C.TARGET, height=C.TARGET, orig_width=native, orig_height=native,
                        gen_model_id=g["model_id"], gen_steps=g_steps, gen_guidance=g_guid,
                        gen_native_res=native, seed=int(seed), caption_model=cfg["caption_model"],
                        pipeline_version=C.PIPELINE_VERSION, sha256=C.sha256_bytes(png),
                        created_utc=C.now_utc()))
        writer.add(row); accepted += 1
        if accepted % 100 == 0:
            print(f"  {MODEL}: accepted {accepted} this run (total ~{len(done)+accepted}/{TARGET}); failed {failed}")
            gc.collect(); torch.cuda.empty_cache()
        writer.maybe_flush()
finally:
    writer.close()
print(f"done. accepted this run: {accepted}, failed: {failed}, total now ~{len(done)+accepted}/{TARGET}")

## Self-check

In [ ]:
# quick self-check for THIS generator
n = C.count_rows(REPO_ID, f"data/{MODEL}/", TOKEN)
print(f"{MODEL}: {n} rows in data/{MODEL}/ (target {cfg['target_per_generator']})")
# decode one to confirm it is a canonical 512 PNG paired to a real id
import random
shards = C.list_shards(REPO_ID, f"data/{MODEL}/", TOKEN)
if shards:
    loc = C.hf_hub_download(REPO_ID, shards[-1], repo_type="dataset", token=TOKEN)
    t = C.pq.read_table(loc, columns=["image", "image_id", "source_real_id", "label"])
    j = random.randrange(t.num_rows)
    ok, why = C.png_is_canonical(t.column("image")[j].as_py())
    print("sample:", t.column("image_id")[j].as_py(), "<-", t.column("source_real_id")[j].as_py(),
          "| label", t.column("label")[j].as_py(), "| canonical", ok, why)
print("RESULT:", "looks correct" if n > 0 else "no rows yet")